# SWaT Poisoning-Robustness — Results Visualization (HPO-Updated)

This is a companion to `SWaT_Results_Visualization_1.ipynb` that **replaces the AE and LSTM-AE rows with the HPO winners' data** while keeping the other nine detectors unchanged. Everything else — the V1-V9 figure set, SOD appendix, academic styling — mirrors the reference notebook.

**What changes vs the reference:**

| Detector | Reference (R01 paper) | HPO-updated (this notebook) |
|---|---|---|
| AE | `[256,128,64]` ReLU BN dp=0.1, Adam, bs=1024, f1_optimal | `[256,128,64]` **LeakyReLU** BN dp=0.1, Adam, **bs=2048**, f1_optimal |
| LSTM-AE | W=20, h=128, 1-layer, Adam, lr=5e-4, bs=512, 50ep | W=**30**, h=**256**, 1-layer, **AdamW**, lr=**1e-3**, bs=**256**, 30ep |
| All 9 others (iforest, svm, lof, cluster, knn, histogram, pca, mcd, abod) | unchanged | unchanged |
| SOD | excluded (inverted score ordering) | excluded |

**Figure output:** `$SWAT_OUTPUT_DIR/figures_v2_hpo/` — separate folder to avoid overwriting the reference `figures_v2_academic/`.

### How to run

```bash
cd ~/projects/def-liyang/$USER/narval_swat_run
source venv/bin/activate
export SWAT_OUTPUT_DIR=/scratch/$USER/swat_paper_run
# open in VS Code / JupyterLab and Run All
```

No GPU required. Runs in seconds on a login node. Figures match the `V*` prefix and academic style of the reference notebook.

## 1. Setup

In [ ]:
import os
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

OUT_DIR = Path(os.environ.get("SWAT_OUTPUT_DIR", "$SCRATCH/swat_paper_run"))
FIG_DIR = OUT_DIR / "figures_v2_hpo"
FIG_DIR.mkdir(parents=True, exist_ok=True)

EXCLUDE_MODELS = {"sod"}

# Academic plot style — identical to SWaT_Results_Visualization_1
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.titlesize": 20,
    "axes.titleweight": "bold",
    "axes.labelsize": 16,
    "axes.labelweight": "bold",
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
    "lines.linewidth": 2.2,
    "lines.markersize": 7,
})

FAMILY = {
    "iforest":    "Tree",
    "svm":        "Kernel",
    "lof":        "Distance/Density",
    "cluster":    "Distance/Density",
    "knn":        "Distance/Density",
    "abod":       "Distance/Density",
    "histogram":  "Statistical",
    "pca":        "Statistical",
    "mcd":        "Statistical",
    "sod":        "Distance/Density",
    "autoencoder":"Neural",
    "lstm_ae":    "Neural",
}
FAMILY_COLORS = {
    "Tree":             "#4C72B0",
    "Kernel":           "#DD8452",
    "Distance/Density": "#C44E52",
    "Statistical":      "#55A868",
    "Neural":           "#8172B2",
}
ATTACK_COLORS = {
    "feature_noise": "#4C72B0",
    "random_flip":   "#DD8452",
    "targeted_flip": "#55A868",
}

MODEL_COLOR_MAP = {}

def format_model_name(name):
    return name.upper() if name != "lstm_ae" else "LSTM-AE"

def format_attack_name(name):
    return name.replace("_", " ").title()

def format_rate_ticks(ax, rates):
    ticks = [0] + list(rates)
    ax.set_xticks(ticks)
    ax.set_xticklabels(["0%"] + [f"{int(r * 100)}%" for r in rates])

def get_model_color_map(models):
    global MODEL_COLOR_MAP
    if not MODEL_COLOR_MAP:
        MODEL_COLOR_MAP = {m: c for m, c in zip(models, plt.cm.tab20(np.linspace(0, 1, len(models))))}
    return MODEL_COLOR_MAP

print(f"Reading R01 paper outputs from: {OUT_DIR}")
print(f"Reading HPO outputs from:        {OUT_DIR}/hpo/results")
print(f"Saving HPO-updated figures to:   {FIG_DIR}")

## 2. Load the R01 paper CSVs + the 18 HPO Final JSONs

We use the R01 paper's `all_results.csv` as the base (gives us all 9 non-AE/non-LSTM detectors), and overlay the HPO Final per-task JSONs to replace the AE and LSTM-AE rows.

In [ ]:
# R01 paper run
all_df_r01 = pd.read_csv(OUT_DIR / "all_results.csv")
print(f"R01 all_results rows: {len(all_df_r01)}")
print(f"R01 models:           {sorted(all_df_r01['model'].unique())}")

# Normalize NaN attack/rate on the R01 clean baselines
all_df_r01["attack"]       = all_df_r01["attack"].fillna("none").astype(str)
all_df_r01["poison_rate"]  = pd.to_numeric(all_df_r01["poison_rate"], errors="coerce").fillna(0.0)

# HPO Final stage — 9 JSONs per detector
HPO_FINAL = {"autoencoder": [], "lstm_ae": []}
for detector_json, detector_main in [("ae", "autoencoder"), ("lstm_ae", "lstm_ae")]:
    d = OUT_DIR / "hpo" / "results" / f"{detector_json}_final"
    for jf in sorted(d.glob("[0-9]*.json")):
        if jf.name.endswith(".error.json"):
            continue
        HPO_FINAL[detector_main].append(json.load(open(jf)))

print(f"HPO autoencoder Final JSONs: {len(HPO_FINAL['autoencoder'])}")
print(f"HPO lstm_ae    Final JSONs: {len(HPO_FINAL['lstm_ae'])}")

## 3. Pick the HPO winning config per detector (multi-seed ranked)

In [ ]:
def _config_key(cfg: dict) -> str:
    return json.dumps({k: v for k, v in cfg.items() if k != "seed"}, sort_keys=True)

def _pick_winner(records):
    """Group by config (ignoring seed), rank by mean composite_f1, return winning cfg_key."""
    buckets = {}
    for r in records:
        buckets.setdefault(_config_key(r["config"]), []).append(r)
    rows = []
    for key, rs in buckets.items():
        cv = [r["results"]["clean"]["f1"] for r in rs]
        pv = [r["results"].get("targeted_flip__r0.10", {}).get("f1", 0.0) for r in rs]
        comp = [(c + p) / 2 for c, p in zip(cv, pv)]
        rows.append((key, float(np.mean(comp)), float(np.std(comp, ddof=1) if len(comp) > 1 else 0)))
    rows.sort(key=lambda r: r[1], reverse=True)
    return rows[0]

WINNERS = {}
for det, records in HPO_FINAL.items():
    key, comp_mean, comp_sd = _pick_winner(records)
    WINNERS[det] = {"key": key, "comp_mean": comp_mean, "comp_sd": comp_sd}
    print(f"{det:<12s} HPO winner composite_f1 = {comp_mean:.4f} ± {comp_sd:.4f}")
    print(f"              config: {json.loads(key)}")

# Select only the Final-stage JSONs that match the winning config for each detector
WINNER_RECORDS = {
    det: [r for r in HPO_FINAL[det] if _config_key(r["config"]) == WINNERS[det]["key"]]
    for det in HPO_FINAL
}
for det, recs in WINNER_RECORDS.items():
    print(f"{det}: {len(recs)} seed-records retained for the winning config")

## 4. Convert HPO Final records → rows matching `all_results.csv` schema

For each detector's winning config, emit 13 rows per seed (1 clean + 3 attacks × 4 rates = 13 conditions) × 3 seeds = 39 rows. Across the two detectors, 78 new rows replace the old R01 AE + LSTM-AE rows.

In [ ]:
def _hpo_rows(detector: str, records: list) -> pd.DataFrame:
    rows = []
    for r in records:
        seed = r["config"]["seed"]
        input_dim = r.get("input_dim", 44)
        n_train = r.get("n_train_normal", 276708)
        for cond_key, block in r["results"].items():
            if cond_key == "clean":
                attack = "none"
                rate = 0.0
            else:
                attack = block.get("poison_attack") or cond_key.split("__r")[0]
                rate   = float(block.get("poison_rate") or cond_key.split("__r")[1])
            rows.append({
                "accuracy":   block.get("accuracy", np.nan),
                "precision":  block.get("precision", np.nan),
                "recall":     block.get("recall", np.nan),
                "f1":         block.get("f1", np.nan),
                "tp":         block.get("tp", np.nan),
                "tn":         block.get("tn", np.nan),
                "fp":         block.get("fp", np.nan),
                "fn":         block.get("fn", np.nan),
                "fnr":        block.get("fnr", np.nan),
                "fpr":        block.get("fpr", np.nan),
                "roc_auc":    block.get("roc_auc", np.nan),
                "pr_auc":     block.get("pr_auc", np.nan),
                "model":      detector,
                "seed":       seed,
                "attack":     attack,
                "poison_rate":rate,
                "time":       block.get("train_time", np.nan),
                "train_normal_size": n_train,
                "split_type": "random" if detector == "autoencoder" else "contiguous_normal",
                "threshold":  block.get("threshold", np.nan),
                "n_features": input_dim,
                "effective_contamination": block.get("effective_contamination", 0.0),
                "n_injected":              block.get("n_injected", 0),
            })
    return pd.DataFrame(rows)

hpo_ae_df   = _hpo_rows("autoencoder", WINNER_RECORDS["autoencoder"])
hpo_lstm_df = _hpo_rows("lstm_ae",     WINNER_RECORDS["lstm_ae"])
print(f"HPO autoencoder rows: {len(hpo_ae_df)}  (expect 3 seeds × 13 conditions = 39)")
print(f"HPO lstm_ae     rows: {len(hpo_lstm_df)} (expect 39)")
print()
print("Sanity check — AE seed 42 clean F1:",
      hpo_ae_df.query("seed == 42 and attack == 'none'")["f1"].iloc[0])
print("LSTM-AE seed 42 targeted_flip @ 10%:",
      hpo_lstm_df.query("seed == 42 and attack == 'targeted_flip' and poison_rate == 0.10")["f1"].iloc[0])

## 5. Merge — drop R01 AE + LSTM-AE rows, concat HPO rows

In [ ]:
NEURAL_MODELS = {"autoencoder", "lstm_ae"}
r01_without_neural = all_df_r01[~all_df_r01["model"].isin(NEURAL_MODELS)].copy()

# Concat — preserves column ordering from the R01 schema
all_df = pd.concat([r01_without_neural, hpo_ae_df, hpo_lstm_df], ignore_index=True)
all_df["attack"]      = all_df["attack"].fillna("none").astype(str)
all_df["poison_rate"] = pd.to_numeric(all_df["poison_rate"], errors="coerce").fillna(0.0)

print(f"Merged all_df rows:    {len(all_df)}")
print(f"  R01 non-neural rows: {len(r01_without_neural)}")
print(f"  HPO AE rows:         {len(hpo_ae_df)}")
print(f"  HPO LSTM-AE rows:    {len(hpo_lstm_df)}")
print(f"  Expected total:      {len(r01_without_neural) + 78}")
print(f"\nModels in merged df:  {sorted(all_df['model'].unique())}")

## 6. Recompute T4 (clean baselines), compute_cost, and ranking from merged data

The R01 `table_T4_clean_baselines.csv`, `compute_cost.csv`, and `multi_criteria_ranking.csv` were computed from the old AE/LSTM-AE data. We recompute them in-memory from the merged DataFrame so downstream plots reflect the HPO winners.

In [ ]:
def _table_T4(df):
    clean = df[df["attack"] == "none"]
    agg = clean.groupby("model").agg(
        f1_mean=("f1","mean"),   f1_std=("f1","std"),
        recall_mean=("recall","mean"),   recall_std=("recall","std"),
        precision_mean=("precision","mean"), precision_std=("precision","std"),
        fnr_mean=("fnr","mean"), fnr_std=("fnr","std"),
        time_mean=("time","mean"),
    ).reset_index()
    return agg

def _compute_cost(df, T4):
    cost = T4[["model", "f1_mean", "time_mean"]].rename(
        columns={"f1_mean":"clean_f1","time_mean":"train_s"})
    cost["verdict"] = cost["train_s"].apply(
        lambda t: "lightweight (<10s)" if t < 10 else
                  "moderate (<100s)"  if t < 100 else "heavy (>100s)"
    )
    return cost

def _composite_rank(df, T4):
    """Mirror multi_criteria_ranking.csv schema: composite + five sub-scores.
    Uses the same weighting as the R01 ranking (weights hardcoded here to match)."""
    weights = {"clean":0.30,"robust":0.25,"safety":0.25,"stability":0.10,"speed":0.10}
    attacks = sorted(df[df["attack"] != "none"]["attack"].unique())
    rates   = sorted(df[df["attack"] != "none"]["poison_rate"].unique())

    rows = []
    for m in T4["model"].unique():
        cf1 = T4.query("model == @m")["f1_mean"].iloc[0]
        # Worst F1 drop across attack × rate pairs (mean over seeds)
        drops = []
        for a in attacks:
            for r in rates:
                sub = df.query("model == @m and attack == @a and abs(poison_rate - @r) < 1e-6")["f1"]
                if len(sub):
                    drops.append(cf1 - sub.mean())
        worst_f1_drop = max(drops) if drops else 0.0
        # Worst FNR across attacks
        fnr_vals = df.query("model == @m and attack != 'none'")["fnr"]
        worst_fnr = fnr_vals.max() if len(fnr_vals) else 1.0
        # Mean seed-stdev of F1 across all (attack,rate) cells
        per_cell_std = (
            df.query("model == @m and attack != 'none'")
              .groupby(["attack","poison_rate"])["f1"].std(ddof=0)
        )
        mean_seed_std = per_cell_std.mean() if len(per_cell_std) else 0.0
        train_s = T4.query("model == @m")["time_mean"].iloc[0]
        rows.append({
            "model": m, "clean_f1": cf1,
            "worst_f1_drop": worst_f1_drop,
            "worst_fnr": worst_fnr,
            "mean_seed_std": mean_seed_std,
            "train_s": train_s,
        })
    rank = pd.DataFrame(rows)

    def norm(col, higher_is_better=True):
        v = rank[col].values.astype(float)
        lo, hi = float(np.nanmin(v)), float(np.nanmax(v))
        if hi - lo < 1e-12:
            return np.zeros_like(v)
        n = (v - lo) / (hi - lo)
        return n if higher_is_better else 1 - n

    rank["score_clean"]     = norm("clean_f1", True)
    rank["score_robust"]    = norm("worst_f1_drop", False)
    rank["score_safety"]    = norm("worst_fnr", False)
    rank["score_stability"] = norm("mean_seed_std", False)
    rank["score_speed"]     = norm("train_s", False)
    rank["composite"] = (
        weights["clean"]     * rank["score_clean"] +
        weights["robust"]    * rank["score_robust"] +
        weights["safety"]    * rank["score_safety"] +
        weights["stability"] * rank["score_stability"] +
        weights["speed"]     * rank["score_speed"]
    )
    return rank.sort_values("composite", ascending=False).reset_index(drop=True)

T4       = _table_T4(all_df)
cost_df  = _compute_cost(all_df, T4)
rank_df  = _composite_rank(all_df, T4)

print("T4 (clean baselines) — recomputed from merged data:")
print(T4.to_string(index=False))
print()
print("Composite ranking — top 5:")
print(rank_df[["model","composite","clean_f1","worst_f1_drop","worst_fnr","train_s"]]
      .head(5).to_string(index=False))

### 6.5. Save HPO-updated tables to disk

Writes five CSVs to `$SWAT_OUTPUT_DIR/hpo_updated/` with an `_hpo` suffix so they don't overwrite the R01 originals. These are the companion tables to the R01 paper artefacts but with AE and LSTM-AE rows replaced by the HPO winners' data.

In [ ]:
# ── Save the HPO-updated merged tables to disk ───────────────────────
# Written to $SWAT_OUTPUT_DIR/hpo_updated/ alongside the original R01 CSVs,
# with an `_hpo.csv` suffix so they don't overwrite anything.

HPO_CSV_DIR = OUT_DIR / "hpo_updated"
HPO_CSV_DIR.mkdir(parents=True, exist_ok=True)

# 1. Full merged all_results — the main deliverable
all_df.to_csv(HPO_CSV_DIR / "all_results_hpo.csv", index=False)

# 2. Clean baselines table (R01 Table V replacement)
T4.to_csv(HPO_CSV_DIR / "table_T4_clean_baselines_hpo.csv", index=False)

# 3. Compute cost per detector
cost_df.to_csv(HPO_CSV_DIR / "compute_cost_hpo.csv", index=False)

# 4. Multi-criteria composite ranking (R01 Table VI / §6.5 replacement)
rank_df.to_csv(HPO_CSV_DIR / "multi_criteria_ranking_hpo.csv", index=False)

# 5. A per-(attack, rate) poisoning-impact summary for quick reviewing
impact_rows = []
for m in all_df["model"].unique():
    clean_f1 = (all_df.query("model == @m and attack == 'none'")["f1"].mean()
                if (all_df["model"] == m).any() else float("nan"))
    for a in sorted(all_df[all_df["attack"] != "none"]["attack"].unique()):
        for r in sorted(all_df[all_df["attack"] != "none"]["poison_rate"].unique()):
            sel = all_df.query(
                "model == @m and attack == @a and abs(poison_rate - @r) < 1e-6")
            if len(sel):
                impact_rows.append({
                    "model": m, "attack": a, "poison_rate": r,
                    "f1_mean":  sel["f1"].mean(),
                    "f1_std":   sel["f1"].std(ddof=0),
                    "fnr_mean": sel["fnr"].mean(),
                    "delta_f1": sel["f1"].mean() - clean_f1,
                    "n_seeds":  len(sel),
                })
impact_df = pd.DataFrame(impact_rows)
impact_df.to_csv(HPO_CSV_DIR / "table_T5_poisoning_impact_hpo.csv", index=False)

# Summary output
print(f"Saved HPO-updated tables to: {HPO_CSV_DIR}")
for p in sorted(HPO_CSV_DIR.glob("*.csv")):
    sz = p.stat().st_size / 1024
    print(f"  {p.name:<40}  {sz:6.1f} KB   ({sum(1 for _ in open(p)) - 1} rows)")

## 7. Apply the SOD filter (main-analysis views only)

In [ ]:
def drop_sod(df, col="model"):
    return df[~df[col].isin(EXCLUDE_MODELS)].copy()

all_df_main = drop_sod(all_df)
T4_main     = drop_sod(T4)
cost_main   = drop_sod(cost_df)
rank_main   = drop_sod(rank_df)

atk_rows = all_df_main[all_df_main["attack"] != "none"]
ATTACKS = sorted(atk_rows["attack"].unique())
RATES   = sorted(atk_rows["poison_rate"].unique())
MODELS  = list(rank_main.sort_values("composite", ascending=False)["model"])

print(f"Main analysis models ({len(MODELS)}): {MODELS}")
print(f"Attacks: {ATTACKS}")
print(f"Poison rates: {RATES}")
print(f"Clean rows: {len(all_df_main[all_df_main['attack'] == 'none'])}")
print(f"Attack rows: {len(atk_rows)}")

## 8. V1 — Clean baseline bar chart

In [ ]:
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(12.5, 7.2))
sub = T4_main.sort_values("f1_mean", ascending=True).reset_index(drop=True)
ypos = np.arange(len(sub))
colors = [FAMILY_COLORS[FAMILY[m]] for m in sub["model"]]

ax.barh(ypos, sub["f1_mean"], xerr=sub["f1_std"], color=colors,
        edgecolor="black", linewidth=0.6,
        error_kw={"elinewidth": 1.1, "capsize": 3, "capthick": 1.1}, zorder=3)
ax.set_yticks(ypos)
ax.set_yticklabels(sub["model"].map(format_model_name), fontsize=15)

text_y_offsets = {"mcd": -0.18, "iforest": 0.02, "cluster": 0.14}
for i, (_, row) in enumerate(sub.iterrows()):
    y = i + text_y_offsets.get(row["model"], 0.0)
    ax.text(row["f1_mean"] + 0.012, y,
            f'F1={row["f1_mean"]:.3f} | FNR={row["fnr_mean"]:.3f}',
            va="center", ha="left", fontsize=12)

ax.set_xlabel("Clean-data F1 (mean ± SD across 3 seeds)", fontsize=18, fontweight="bold")
ax.set_ylabel("Model", fontsize=18, fontweight="bold")
ax.set_xlim(0, 1.12)
ax.set_title("Clean-Baseline Performance (HPO-Updated)", fontsize=24, fontweight="bold")
ax.grid(axis="x", linestyle="--", alpha=0.3, zorder=0)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.tick_params(axis="x", labelsize=14); ax.tick_params(axis="y", labelsize=14)

handles = [Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()
           if any(FAMILY[m] == f for m in sub["model"])]
ax.legend(handles=handles, title="Model Family", title_fontsize=12, fontsize=11,
          loc="lower right", bbox_to_anchor=(-0.010, -0.35), frameon=False)

plt.tight_layout()
plt.savefig(FIG_DIR / "V1_clean_baseline_bar.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. V2 — Cost vs clean F1 scatter

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6.5))
label_offsets = {"autoencoder": (-50, 6), "abod": (10, 12), "knn": (10, -2),
                 "lstm_ae": (8, 8), "lof": (8, 6)}

for _, r in cost_main.iterrows():
    m = r["model"]; color = FAMILY_COLORS[FAMILY[m]]
    ax.scatter(r["train_s"], r["clean_f1"], s=140, color=color,
               edgecolor="black", linewidth=0.6, zorder=3)
    dx, dy = label_offsets.get(m, (8, 7))
    ax.annotate(format_model_name(m), (r["train_s"], r["clean_f1"]),
                xytext=(dx, dy), textcoords="offset points", fontsize=11)

ax.set_xscale("log")
ax.set_xlabel("Training time (s, log scale)", fontsize=15, fontweight="bold")
ax.set_ylabel("Clean-data F1", fontsize=15, fontweight="bold")
ax.set_title("Training Cost vs Clean Performance (HPO-Updated)", fontsize=20, fontweight="bold")
ax.axhspan(0.85, 1.0, alpha=0.04, color="green")
ax.axvspan(0.1, 10, alpha=0.04, color="green")
ax.text(1.2, 0.885, "Faster and stronger region", fontsize=11, alpha=0.75, style="italic")
ax.tick_params(axis="both", labelsize=12); ax.grid(True, alpha=0.25, zorder=0)
plt.tight_layout()
plt.savefig(FIG_DIR / "V2_cost_vs_f1_scatter.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. V3 — Multi-criteria composite ranking

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.5))
sub = rank_main.sort_values("composite", ascending=True).reset_index(drop=True)
colors = [FAMILY_COLORS[FAMILY[m]] for m in sub["model"]]

ax.barh(sub["model"].map(format_model_name), sub["composite"],
        color=colors, edgecolor="black", linewidth=0.5)
for i, v in enumerate(sub["composite"]):
    ax.text(v + 0.008, i, f"{v:.3f}", va="center", fontsize=11)
ax.set_xlabel("Composite score"); ax.set_ylabel("Model")
ax.set_xlim(0, 1.0)
ax.set_title("Multi-Criteria Ranking (HPO-Updated)")
plt.tight_layout()
plt.savefig(FIG_DIR / "V3_composite_ranking.png", bbox_inches="tight")
plt.show()

### 10b. V3b — Ranking components stacked

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
sub = rank_main.sort_values("composite", ascending=False).reset_index(drop=True)
components = ["score_clean", "score_robust", "score_safety", "score_stability", "score_speed"]
weights = {"score_clean": 0.30, "score_robust": 0.25, "score_safety": 0.25,
           "score_stability": 0.10, "score_speed": 0.10}
labels = {"score_clean": "Clean F1", "score_robust": "Robustness",
          "score_safety": "Low FNR", "score_stability": "Seed Stability",
          "score_speed": "Training Speed"}
palette = plt.cm.viridis(np.linspace(0.1, 0.95, len(components)))

bottom = np.zeros(len(sub))
xlabels = sub["model"].map(format_model_name)
for comp, col in zip(components, palette):
    contribution = sub[comp] * weights[comp]
    ax.bar(xlabels, contribution, bottom=bottom, label=labels[comp],
           color=col, edgecolor="white", linewidth=0.5)
    bottom = bottom + contribution.values

ax.set_ylabel("Weighted contribution")
ax.set_title("Composite Score Decomposition (HPO-Updated)")
ax.legend(loc="upper right", frameon=False, ncol=2)
plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "V3b_ranking_stacked.png", bbox_inches="tight")
plt.show()

## 11. V4 — Robustness curves (F1 vs poison rate, per attack)

In [ ]:
fig, axes = plt.subplots(len(ATTACKS), 1, figsize=(16, 16), sharey=True)
if len(ATTACKS) == 1: axes = [axes]

m_color = get_model_color_map(MODELS)
legend_handles = {}
for idx, (ax, atk) in enumerate(zip(axes, ATTACKS)):
    for m in MODELS:
        clean = T4_main[T4_main["model"] == m]["f1_mean"].iloc[0]
        xs, ys = [0], [clean]
        for r in RATES:
            sel = all_df_main[(all_df_main["model"] == m) &
                              (all_df_main["attack"] == atk) &
                              (np.isclose(all_df_main["poison_rate"], r))]
            if len(sel):
                xs.append(r); ys.append(sel["f1"].mean())
        line, = ax.plot(xs, ys, "-o", color=m_color[m], label=format_model_name(m))
        if m not in legend_handles: legend_handles[m] = line
    ax.set_title(format_attack_name(atk))
    ax.set_ylabel("F1 Score")
    format_rate_ticks(ax, RATES)
    ax.set_ylim(-0.05, 1.05)
    if idx == len(ATTACKS) - 1: ax.set_xlabel("Poison Rate")

fig.suptitle("F1 vs Poison Rate (HPO-Updated)", fontsize=24, fontweight="bold")
fig.legend(handles=list(legend_handles.values()),
           labels=[format_model_name(m) for m in legend_handles.keys()],
           loc="center right", bbox_to_anchor=(0.99, 0.5),
           frameon=False, ncol=1, fontsize=12)
plt.tight_layout(rect=[0, 0, 0.83, 0.96])
plt.savefig(FIG_DIR / "V4_robustness_curves.png", bbox_inches="tight")
plt.show()

## 12. V5 — F1-degradation heatmap

In [ ]:
cols = [(a, r) for a in ATTACKS for r in RATES]
mat = np.full((len(MODELS), len(cols)), np.nan)
for i, m in enumerate(MODELS):
    cf1 = T4_main[T4_main["model"] == m]["f1_mean"].iloc[0]
    for j, (a, r) in enumerate(cols):
        sel = all_df_main[(all_df_main["model"] == m) &
                          (all_df_main["attack"] == a) &
                          (np.isclose(all_df_main["poison_rate"], r))]
        if len(sel):
            mat[i, j] = sel["f1"].mean() - cf1

attack_short = {"feature_noise": "Feature Noise", "random_flip": "Random Flip",
                "targeted_flip": "Targeted Flip"}

fig, ax = plt.subplots(figsize=(16, 7.5))
vmax = max(0.1, np.nanmax(np.abs(mat)))
im = ax.imshow(mat, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
ax.set_yticks(range(len(MODELS)))
ax.set_yticklabels([format_model_name(m) for m in MODELS], fontsize=14)
ax.set_xticks(range(len(cols)))
ax.set_xticklabels([f"{attack_short.get(a, format_attack_name(a))}\n{int(r*100)}%" for a, r in cols],
                    rotation=35, ha="right", rotation_mode="anchor", fontsize=10)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat[i, j]
        if pd.notna(v):
            color = "white" if abs(v) > 0.22 else "black"
            ax.text(j, i, f"{v:+.2f}", ha="center", va="center", fontsize=8.5, color=color)

group_size = len(RATES)
for k in range(1, len(ATTACKS)):
    ax.axvline(k * group_size - 0.5, color="gray", linewidth=1.2, alpha=0.7)

cbar = plt.colorbar(im, ax=ax, shrink=0.9)
cbar.set_label("ΔF1 (poisoned − clean)", fontsize=13, fontweight="bold")
cbar.ax.tick_params(labelsize=12)
ax.set_title("F1 Change Across Attack Scenarios (HPO-Updated)", fontsize=20, fontweight="bold")
ax.set_xlabel("Attack Type and Poison Rate", fontsize=16, fontweight="bold")
ax.set_ylabel("Model", fontsize=16, fontweight="bold")
ax.tick_params(axis="x", pad=8); ax.tick_params(axis="y", labelsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / "V5_degradation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## 13. V6 — FNR safety plot

In [ ]:
fig, axes = plt.subplots(len(ATTACKS), 1, figsize=(16, 16), sharey=True)
if len(ATTACKS) == 1: axes = [axes]
m_color = get_model_color_map(MODELS)
legend_handles = {}
for idx, (ax, atk) in enumerate(zip(axes, ATTACKS)):
    for m in MODELS:
        clean_fnr = T4_main[T4_main["model"] == m]["fnr_mean"].iloc[0]
        xs, ys = [0], [clean_fnr]
        for r in RATES:
            sel = all_df_main[(all_df_main["model"] == m) &
                              (all_df_main["attack"] == atk) &
                              (np.isclose(all_df_main["poison_rate"], r))]
            if len(sel):
                xs.append(r); ys.append(sel["fnr"].mean())
        line, = ax.plot(xs, ys, "-o", color=m_color[m], label=format_model_name(m))
        if m not in legend_handles: legend_handles[m] = line
    ax.axhspan(0.5, 1.0, alpha=0.06, color="red")
    ax.set_title(format_attack_name(atk))
    ax.set_ylabel("False-Negative Rate")
    format_rate_ticks(ax, RATES)
    ax.set_ylim(-0.02, 1.02)
    if idx == len(ATTACKS) - 1: ax.set_xlabel("Poison Rate")

fig.suptitle("False-Negative Rate vs Poison Rate (HPO-Updated)", fontsize=24, fontweight="bold")
fig.legend(handles=list(legend_handles.values()),
           labels=[format_model_name(m) for m in legend_handles.keys()],
           loc="center right", bbox_to_anchor=(0.99, 0.5),
           frameon=False, fontsize=12)
plt.tight_layout(rect=[0, 0, 0.83, 0.96])
plt.savefig(FIG_DIR / "V6_fnr_safety.png", bbox_inches="tight")
plt.show()

## 14. V7 — Seed-variance bands

In [ ]:
fig, axes = plt.subplots(len(ATTACKS), 1, figsize=(16, 16), sharey=True)
if len(ATTACKS) == 1: axes = [axes]
m_color = get_model_color_map(MODELS)
legend_handles = {}
for idx, (ax, atk) in enumerate(zip(axes, ATTACKS)):
    for m in MODELS:
        clean = T4_main[T4_main["model"] == m]["f1_mean"].iloc[0]
        clean_std = T4_main[T4_main["model"] == m]["f1_std"].iloc[0]
        xs, mu, sd = [0], [clean], [clean_std]
        for r in RATES:
            sel = all_df_main[(all_df_main["model"] == m) &
                              (all_df_main["attack"] == atk) &
                              (np.isclose(all_df_main["poison_rate"], r))]["f1"]
            if len(sel):
                xs.append(r); mu.append(sel.mean()); sd.append(sel.std(ddof=0))
        xs = np.array(xs); mu = np.array(mu); sd = np.nan_to_num(np.array(sd))
        color = m_color[m]
        line, = ax.plot(xs, mu, "-o", color=color, label=format_model_name(m))
        ax.fill_between(xs, mu - sd, mu + sd, color=color, alpha=0.12)
        if m not in legend_handles: legend_handles[m] = line
    ax.set_title(format_attack_name(atk))
    ax.set_ylabel("F1 Score")
    format_rate_ticks(ax, RATES)
    ax.set_ylim(-0.05, 1.05)
    if idx == len(ATTACKS) - 1: ax.set_xlabel("Poison Rate")

fig.suptitle("F1 with Seed Variability Bands (HPO-Updated)", fontsize=24, fontweight="bold")
fig.legend(handles=list(legend_handles.values()),
           labels=[format_model_name(m) for m in legend_handles.keys()],
           loc="center right", bbox_to_anchor=(0.99, 0.5),
           frameon=False, fontsize=12)
plt.tight_layout(rect=[0, 0, 0.83, 0.96])
plt.savefig(FIG_DIR / "V7_seed_variance.png", bbox_inches="tight")
plt.show()

## 15. V8 — Per-model comparison grid

In [ ]:
ncol = 4
nrow = int(np.ceil(len(MODELS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(18, 12), sharey=True, sharex=True)
axes = axes.flatten()

for i, m in enumerate(MODELS):
    ax = axes[i]
    clean = T4_main[T4_main["model"] == m]["f1_mean"].iloc[0]
    for atk in ATTACKS:
        xs, ys = [0], [clean]
        for r in RATES:
            sel = all_df_main[(all_df_main["model"] == m) &
                              (all_df_main["attack"] == atk) &
                              (np.isclose(all_df_main["poison_rate"], r))]
            if len(sel):
                xs.append(r); ys.append(sel["f1"].mean())
        ax.plot(xs, ys, "-o", color=ATTACK_COLORS[atk], label=format_attack_name(atk),
                linewidth=2.0, markersize=5)
    ax.set_title(format_model_name(m), fontsize=14, fontweight="bold")
    format_rate_ticks(ax, RATES)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.25)

for j in range(len(MODELS), len(axes)):
    axes[j].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.05),
           ncol=3, frameon=False, fontsize=12)
fig.supxlabel("Poison Rate", fontsize=16, fontweight="bold")
fig.supylabel("F1 Score", fontsize=16, fontweight="bold")
fig.suptitle("Per-Model Comparison Across Attack Types (HPO-Updated)",
             fontsize=22, fontweight="bold")
plt.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.savefig(FIG_DIR / "V8_per_model_grid.png", bbox_inches="tight")
plt.show()

## 16. V9 — Worst-case F1 degradation

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.5))
sub = rank_main.sort_values("worst_f1_drop", ascending=True).reset_index(drop=True)
colors = [FAMILY_COLORS[FAMILY[m]] for m in sub["model"]]
ax.barh(sub["model"].map(format_model_name), sub["worst_f1_drop"],
        color=colors, edgecolor="black", linewidth=0.5)
for i, v in enumerate(sub["worst_f1_drop"]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=11)
ax.set_xlabel("Worst-case F1 drop across all attack-rate pairs")
ax.set_ylabel("Model")
ax.set_title("Worst-Case Robustness (HPO-Updated)")
ax.axvline(0.10, color="gray", linestyle=":", linewidth=1.4, label="0.10 reference")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "V9_worst_case_drop.png", bbox_inches="tight")
plt.show()

## 17. NEW — side-by-side comparison with R01 deployed config (key takeaway)

A single figure that highlights exactly what the HPO bought us for the two neural detectors: clean F1 equivalence + dramatically smaller poisoned-F1 drops.

In [ ]:
# R01 reference numbers for AE and LSTM-AE (from the original all_results.csv before overlay)
r01_neural = all_df_r01[all_df_r01["model"].isin({"autoencoder", "lstm_ae"})].copy()
r01_neural["attack"] = r01_neural["attack"].fillna("none").astype(str)
r01_neural["poison_rate"] = pd.to_numeric(r01_neural["poison_rate"], errors="coerce").fillna(0.0)

def _summary(df, m):
    clean = df[(df["model"] == m) & (df["attack"] == "none")]["f1"]
    pois = df[(df["model"] == m) & (df["attack"] == "targeted_flip") &
              (np.isclose(df["poison_rate"], 0.10))]["f1"]
    return (clean.mean(), clean.std(ddof=0), pois.mean(), pois.std(ddof=0))

summaries = []
for m in ["autoencoder", "lstm_ae"]:
    cm_r, cs_r, pm_r, ps_r = _summary(r01_neural, m)
    cm_h, cs_h, pm_h, ps_h = _summary(all_df,     m)
    summaries.append((m, cm_r, cs_r, pm_r, ps_r, cm_h, cs_h, pm_h, ps_h))

fig, ax = plt.subplots(figsize=(11, 6.5))
labels = ["AE\n(R01)", "AE\n(HPO)", "LSTM-AE\n(R01)", "LSTM-AE\n(HPO)"]
x = np.arange(len(labels)); width = 0.35

clean_m = [summaries[0][1], summaries[0][5], summaries[1][1], summaries[1][5]]
clean_s = [summaries[0][2], summaries[0][6], summaries[1][2], summaries[1][6]]
pois_m  = [summaries[0][3], summaries[0][7], summaries[1][3], summaries[1][7]]
pois_s  = [summaries[0][4], summaries[0][8], summaries[1][4], summaries[1][8]]

b1 = ax.bar(x - width/2, clean_m, width, yerr=clean_s, label="Clean F1",
            color="#4C72B0", edgecolor="black", linewidth=0.6, capsize=4, zorder=3)
b2 = ax.bar(x + width/2, pois_m,  width, yerr=pois_s,  label="Poisoned F1 (10% targeted)",
            color="#C44E52", edgecolor="black", linewidth=0.6, capsize=4, zorder=3)
for b, v in zip(b1, clean_m):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.3f}", ha="center", fontsize=11)
for b, v in zip(b2, pois_m):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.3f}", ha="center", fontsize=11)

ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=13)
ax.set_ylabel("F1 Score"); ax.set_ylim(0, 1.05)
ax.set_title("V10 — R01 Deployed vs HPO Winner (AE + LSTM-AE)")
ax.legend(loc="upper right", frameon=False)
ax.grid(axis="y", linestyle="--", alpha=0.3, zorder=0)
ax.axvline(1.5, color="gray", linewidth=0.8, alpha=0.5)
plt.tight_layout()
plt.savefig(FIG_DIR / "V10_r01_vs_hpo.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nDelta summary:")
for m, cm_r, cs_r, pm_r, ps_r, cm_h, cs_h, pm_h, ps_h in summaries:
    print(f"  {m}: clean {cm_r:.3f}±{cs_r:.3f} → {cm_h:.3f}±{cs_h:.3f}   "
          f"poisoned {pm_r:.3f}±{ps_r:.3f} → {pm_h:.3f}±{ps_h:.3f}")

## 18. Appendix — why SOD is excluded

Same appendix cell as the reference notebook — kept here for completeness.

In [ ]:
T4_full = _table_T4(all_df)  # includes SOD if present
sod_clean = T4_full[T4_full["model"] == "sod"]
sod_attack = all_df[all_df["model"] == "sod"]
print("SOD clean baseline (seed 42 only):"); print(sod_clean.to_string(index=False))
print(f"\nSOD attack runs on record: {len(sod_attack)} (all excluded from main analyses)")

compare = pd.concat([
    T4_full[T4_full["model"] == "sod"][["model","f1_mean","fnr_mean","time_mean"]],
    T4_main.sort_values("f1_mean", ascending=False).head(3)[["model","f1_mean","fnr_mean","time_mean"]]
]).reset_index(drop=True)
compare.columns = ["model", "clean F1", "clean FNR", "training time (s)"]
print("\nSOD alongside the three best clean-F1 detectors (HPO-Updated set):")
print(compare.round(4).to_string(index=False))

## 19. σ-ablation — feature-noise injection sensitivity (existing data, R01 configs)

We have a 165-run σ-ablation on disk from the main paper run that swept the feature-noise standard deviation σ ∈ {0.15, 0.30, 0.50, 1.00} at poison rate 0.10 across all 11 detectors and 3 seeds. The σ=1.00 case additionally has a rate sweep at {0.01, 0.03, 0.05, 0.10}.

**Important caveat:** these AE and LSTM-AE rows use the **R01-deployed configurations**, not the HPO winners. Re-running the σ-ablation under the HPO-winning configs would take ~18 GPU-tasks (2 detectors × 3 seeds × 3 σ values) and is documented in §22 below as a follow-up. For now this section reports the existing data with the caveat marked clearly in plot titles.

In [ ]:
# Load the aggregated σ-sweep CSV (rate=0.10 only) AND the per-task JSONs
# (which include the σ=1.00 rate sweep at {0.01, 0.03, 0.05}).
sigma_csv  = OUT_DIR / "sigma_sweep_all_results.csv"
sigma_root = OUT_DIR / "checkpoints" / "sigma_sweep"

if sigma_csv.exists():
    sigma_df = pd.read_csv(sigma_csv)
    print(f"Loaded {len(sigma_df)} rows from sigma_sweep_all_results.csv")
else:
    sigma_df = pd.DataFrame()
    print("WARNING: sigma_sweep_all_results.csv not found — σ-ablation cells will be empty.")

# Recover the σ=1.00 rate-sweep data directly from per-task JSONs (not in the
# aggregated CSV by default — the aggregator filtered to rate=0.10).
rate_sweep_rows = []
if (sigma_root / "s1p00").exists():
    for jf in sorted((sigma_root / "s1p00").glob("*.json")):
        if jf.name.endswith(".error.json"):
            continue
        d = json.load(open(jf))
        rate_sweep_rows.append({
            "model":       d["model"],
            "seed":        d["seed"],
            "noise_sigma": d.get("noise_sigma", 1.0),
            "poison_rate": d["poison_rate"],
            "f1":          d["f1"],
            "fnr":         d["fnr"],
        })
    sigma_full = pd.DataFrame(rate_sweep_rows)
    print(f"Recovered {len(sigma_full)} per-task σ=1.00 JSONs covering rates {sorted(sigma_full['poison_rate'].unique())}")
else:
    sigma_full = pd.DataFrame()
    print("No per-task σ-sweep JSONs found.")

# Filter SOD out of σ-sweep too (consistent with the rest of the notebook)
sigma_main = sigma_df[~sigma_df["model"].isin(EXCLUDE_MODELS)].copy()
sigma_full_main = sigma_full[~sigma_full["model"].isin(EXCLUDE_MODELS)].copy() if len(sigma_full) else sigma_full

### 19a. V11 — F1 vs σ at fixed rate=0.10 (per detector)

Shows which detectors are sensitive to *amplitude* of feature-noise poisoning. LSTM-AE is the standout — the σ=1.00 cliff is visible as a sharp drop in mean F1 with very wide error bars (the seed-bimodal effect).

In [ ]:
# Optionally load HPO-winner σ-sweep (from scripts/aggregate_sigma_hpo.py)
hpo_sigma_csv = OUT_DIR / "hpo" / "results" / "sigma_sweep_hpo_all_results.csv"
if hpo_sigma_csv.exists():
    sigma_hpo_df = pd.read_csv(hpo_sigma_csv)
    print(f"Loaded {len(sigma_hpo_df)} rows of HPO-winner σ-sweep data")
else:
    sigma_hpo_df = pd.DataFrame()
    print("No HPO-winner σ-sweep file yet — run slurm/70_sigma_hpo.sh to produce it.")

if len(sigma_main):
    fig, ax = plt.subplots(figsize=(13, 7))
    SIGMAS = sorted(sigma_main["noise_sigma"].unique())
    sigma_models = sorted(sigma_main["model"].unique())
    m_color = get_model_color_map(sigma_models)

    # R01 curves — solid lines
    for m in sigma_models:
        means, stds = [], []
        for s in SIGMAS:
            sel = sigma_main[(sigma_main["model"] == m) &
                              (np.isclose(sigma_main["noise_sigma"], s))]["f1"]
            means.append(sel.mean() if len(sel) else np.nan)
            stds.append(sel.std(ddof=0) if len(sel) > 1 else 0.0)
        ax.errorbar(SIGMAS, means, yerr=stds, fmt="-o",
                    color=m_color[m], capsize=3, label=format_model_name(m),
                    linewidth=2.0, markeredgecolor="black")

    # HPO curves — dashed lines, only for AE and LSTM-AE
    if len(sigma_hpo_df):
        hpo_at_r10 = sigma_hpo_df[np.isclose(sigma_hpo_df["poison_rate"], 0.10)]
        for m in ["autoencoder", "lstm_ae"]:
            sub = hpo_at_r10[hpo_at_r10["model"] == m]
            if len(sub) == 0:
                continue
            means, stds = [], []
            for s in SIGMAS:
                sel = sub[np.isclose(sub["noise_sigma"], s)]["f1"]
                means.append(sel.mean() if len(sel) else np.nan)
                stds.append(sel.std(ddof=0) if len(sel) > 1 else 0.0)
            ax.errorbar(SIGMAS, means, yerr=stds, fmt="--D",
                        color=m_color[m], capsize=3,
                        label=f"{format_model_name(m)} (HPO)",
                        linewidth=2.0, markeredgecolor="black", alpha=0.85)

    ax.set_xlabel("Feature-noise σ (poison rate fixed at 10%)",
                  fontsize=15, fontweight="bold")
    ax.set_ylabel("F1 Score (mean ± SD across seeds)",
                  fontsize=15, fontweight="bold")
    suffix = " — HPO overlay" if len(sigma_hpo_df) else " (R01 configs)"
    ax.set_title(f"V11 — F1 vs Feature-Noise σ{suffix}",
                 fontsize=18, fontweight="bold")
    ax.set_xscale("log")
    ax.set_xticks(SIGMAS)
    ax.set_xticklabels([f"{s}" for s in SIGMAS])
    ax.minorticks_off()
    ax.set_ylim(0, 1.05)
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5),
              frameon=False, ncol=1)
    ax.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "V11_sigma_sweep.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("σ-sweep data not loaded — skipping V11.")

### 19b. V12 — LSTM-AE rate-gated σ=1.00 cliff

Zooms into the LSTM-AE σ=1.00 condition across poison rates {0.01, 0.03, 0.05, 0.10}. At σ=1.00 the noise amplitude itself isn't enough — the LSTM-AE only collapses when *both* σ and rate are high. This is one of the paper's main findings (§6.2.7 in R01).

In [ ]:
if len(sigma_full_main) and (sigma_full_main["model"] == "lstm_ae").any():
    sub = sigma_full_main[sigma_full_main["model"] == "lstm_ae"].copy()
    rates = sorted(sub["poison_rate"].unique())
    fig, ax = plt.subplots(figsize=(11, 6.5))

    # R01 config: all-seed mean line with jittered seed dots
    means = [sub[np.isclose(sub["poison_rate"], r)]["f1"].mean() for r in rates]
    stds  = [sub[np.isclose(sub["poison_rate"], r)]["f1"].std(ddof=0) for r in rates]
    ax.errorbar(rates, means, yerr=stds, fmt="-o", color="#C44E52",
                linewidth=3, markersize=10, capsize=4,
                markeredgecolor="black",
                label="LSTM-AE R01 (mean ± SD)")
    for seed in sorted(sub["seed"].unique()):
        seed_sub = sub[sub["seed"] == seed].sort_values("poison_rate")
        ax.plot(seed_sub["poison_rate"], seed_sub["f1"], "o",
                color="gray", alpha=0.55, markersize=7)
        for _, r in seed_sub.iterrows():
            ax.annotate(str(int(seed)),
                        (r["poison_rate"], r["f1"]),
                        xytext=(7, 0), textcoords="offset points",
                        fontsize=8, color="#444")

    # HPO overlay — same LSTM-AE but with the HPO-winning config
    hpo_lstm = sigma_hpo_df if len(sigma_hpo_df) else pd.DataFrame()
    if len(hpo_lstm):
        hpo_lstm = hpo_lstm[
            (hpo_lstm["model"] == "lstm_ae") &
            (np.isclose(hpo_lstm["noise_sigma"], 1.00))
        ]
        if len(hpo_lstm):
            hpo_rates = sorted(hpo_lstm["poison_rate"].unique())
            hpo_means = [hpo_lstm[np.isclose(hpo_lstm["poison_rate"], r)]["f1"].mean()
                         for r in hpo_rates]
            hpo_stds  = [hpo_lstm[np.isclose(hpo_lstm["poison_rate"], r)]["f1"].std(ddof=0)
                         for r in hpo_rates]
            ax.errorbar(hpo_rates, hpo_means, yerr=hpo_stds, fmt="--D",
                        color="#4C72B0", linewidth=3, markersize=10, capsize=4,
                        markeredgecolor="black",
                        label="LSTM-AE HPO (mean ± SD)")

    ax.set_xlabel("Poison rate (σ fixed at 1.00)", fontsize=14, fontweight="bold")
    ax.set_ylabel("LSTM-AE F1", fontsize=14, fontweight="bold")
    ax.set_xticks(rates)
    ax.set_xticklabels([f"{int(r*100)}%" for r in rates])
    suffix = " — R01 vs HPO" if len(sigma_hpo_df) else " (R01 config)"
    ax.set_title(f"V12 — LSTM-AE σ=1.00 Cliff Is Rate-Gated{suffix}",
                 fontsize=18, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.axhspan(0, 0.5, alpha=0.05, color="red")
    ax.text(0.005, 0.05, "catastrophic region", fontsize=10,
            color="#8B0000", style="italic")
    ax.legend(loc="lower left", frameon=False, fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "V12_sigma1_rate_cliff.png", dpi=300, bbox_inches="tight")
    plt.show()

    print("\nR01 LSTM-AE per-(seed, rate) F1 at σ=1.00:")
    pivot = (sub.pivot_table(index="seed", columns="poison_rate",
                              values="f1", aggfunc="mean").round(4))
    print(pivot.to_string())

    if len(sigma_hpo_df) and (sigma_hpo_df["model"] == "lstm_ae").any():
        hpo_sub = sigma_hpo_df[
            (sigma_hpo_df["model"] == "lstm_ae") &
            (np.isclose(sigma_hpo_df["noise_sigma"], 1.00))]
        print("\nHPO LSTM-AE per-(seed, rate) F1 at σ=1.00:")
        hpo_pivot = (hpo_sub.pivot_table(index="seed", columns="poison_rate",
                                           values="f1", aggfunc="mean").round(4))
        print(hpo_pivot.to_string())
else:
    print("No LSTM-AE σ=1.00 rate-sweep data found.")

## 22. Done — what was saved

In [ ]:
saved = sorted(FIG_DIR.glob("V*.png"))
print(f"Saved {len(saved)} HPO-updated figures to {FIG_DIR}:")
for p in saved:
    sz = p.stat().st_size / 1024
    print(f"  {p.name:<42}  ({sz:.0f} KB)")